In [0]:
dbutils.widgets.text("url", "https://realtime.hsl.fi/realtime/trip-updates/v2/hsl")
dbutils.widgets.text("message_type", "trip_updates")

url = dbutils.widgets.get("url")
message_type = dbutils.widgets.get("message_type")

In [0]:
from pyspark.sql.protobuf.functions import from_protobuf
descriptor_path = "/Volumes/shared/john_armstrong/gtfs_rt/gtfs_rt.desc"
message_name = "transit_realtime.FeedMessage"  # Full package name from gtfs-realtime.proto

In [0]:
import requests
from pyspark.sql.datasource import SimpleDataSourceStreamReader, DataSource
from pyspark.sql.types import (
    StructType, StructField, BinaryType, StringType
)
from typing import Iterator, Tuple
import time


class GTFSRTSimpleStreamReader(SimpleDataSourceStreamReader):

    def initialOffset(self):
        return {"offset": int(time.time())}

    def __init__(self, schema: StructType, options: dict):
        """Initialize with schema and options."""
        super().__init__()
        self.schema = schema
        self.url = options.get("url", "")
        self.frequency = options.get("frequency", "minutely")

    def read(self, start: dict):
        """Reads data starting from the given offset."""
        data = []
        new_offset = {}
        gtrfsrt = self._fetch_gtfs_rt_data(self.url)
        data.append((self.url, gtrfsrt))
        new_offset.update({"offset": int(time.time())})
        return (data, new_offset)
    
    # def readBetweenOffsets(self, start: dict, end: dict) -> Iterator[Tuple]:
    #     """
    #     Takes start and end offset as inputs, then reads an iterator of data deterministically.
    #     This is called when the query replays batches during restart or after a failure.
    #     """
    #     start_idx = start["offset"]
    #     end_idx = end["offset"]
    #     return iter([(i,) for i in range(start_idx, end_idx)])

    @staticmethod
    def _fetch_gtfs_rt_data(url):        
        try:
            # Send a GET request to the endpoint
            response = requests.get(url)
            response.raise_for_status()  # Raise an exception for HTTP errors (e.g., 404, 500)
            return response.content
        except requests.exceptions.RequestException as e:
            raise e

class GTFSRTDataSource(DataSource):
    @classmethod
    def name(cls):
        """Returns the name of the data source."""
        return "gtfsrt"

    def __init__(self, options):
        """Initialize with options provided."""
        self.options = options

    def schema(self):
        """Returns the schema of the data source."""
        return StructType([
            StructField("url", StringType(), False),
            StructField("binary_proto", BinaryType(), True)
        ])


    def simpleStreamReader(self, schema: StructType):
        """Returns an instance of the reader for this data source."""
        return GTFSRTSimpleStreamReader(schema, self.options)
    
def test():
    return 1

In [0]:
spark.dataSource.register(GTFSRTDataSource)

In [0]:
# Use the streaming data source
df = (spark.readStream
      .format("gtfsrt")
      .option("url", url)
      .load())

In [0]:
# Deserialize protobuf data using native from_protobuf function and the descriptor file
proto_df = df.withColumn(
    "parsed_feed_message", from_protobuf("binary_proto", message_name, descFilePath=descriptor_path).alias("proto_data")
)

In [0]:
# Sink results in to Delta Table
(
    proto_df
    .writeStream
    .outputMode("append")
    .option("checkpointLocation", f"/dbfs/john_armstrong/checkpoints/gtfsrt/{message_type}")
    .trigger(processingTime="10 seconds")
    .toTable("shared.john_armstrong.gtfs_rt")
)

In [0]:
def fetch(url):        
    try:
        # Send a GET request to the endpoint
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for HTTP errors (e.g., 404, 500)
        return response.content
    except requests.exceptions.RequestException as e:
        raise e

In [0]:
binary_data = fetch(url)

In [0]:
# Write binary data to a file
import os
os.makedirs("/Volumes/shared/john_armstrong/gtfs_rt/test/", exist_ok=True)

with open("/Volumes/shared/john_armstrong/gtfs_rt/test/data.bin", "wb") as f:
    f.write(binary_data)

In [0]:
df = spark.read.format("binaryFile").load("/Volumes/shared/john_armstrong/gtfs_rt/test/data.bin")

In [0]:
display(df)

In [0]:
proto = df.withColumn("proto", from_protobuf("content", message_name, descFilePath=descriptor_path))

In [0]:
df.write.save("adls://john-armstrong@john-armstrong.azuredatalakestore.net/john_armstrong/test/")

In [0]:
display(proto.select("proto.header"))